# **Import Package**

In [1]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Cek Akselerasi CUDA GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sub-models akan dilatih menggunakan: {device}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

# Path basis untuk data processed sub-models
BASE_SUB_DIR = "../data/processed/level_2_sub"

Sub-models akan dilatih menggunakan: cuda
Active GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


# **Augmentasi**

In [3]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val_test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}
BATCH_SIZE = 32

# **Sub Model Organik(MobileNetV3)**

### **Load Data Organik**

In [5]:
org_dir = os.path.join(BASE_SUB_DIR, "organic_branch")
train_dataset_org = datasets.ImageFolder(os.path.join(org_dir, "train"), data_transforms['train'])
val_dataset_org = datasets.ImageFolder(os.path.join(org_dir, "val"), data_transforms['val_test'])

train_loader_org = DataLoader(train_dataset_org, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader_org = DataLoader(val_dataset_org, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

org_classes = train_dataset_org.classes
print(f"Kelas Cabang Organik ({len(org_classes)} kelas): {org_classes}")

Kelas Cabang Organik (2 kelas): ['Food Organics', 'Vegetation']


### **Build Model**

In [6]:
print("Memuat pre-trained MobileNetV3-Large...")
model_organic = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
num_features_org = model_organic.classifier[3].in_features
model_organic.classifier[3] = nn.Linear(num_features_org, len(org_classes))
model_organic = model_organic.to(device)

Memuat pre-trained MobileNetV3-Large...


### **Loss dan Optimizer**

In [7]:
criterion_org = nn.CrossEntropyLoss()
optimizer_org = optim.Adam(model_organic.parameters(), lr=1e-4)

### **Train Function**

In [8]:
EPOCHS_ORG = 10
best_acc_org = 0.0
best_wts_org = copy.deepcopy(model_organic.state_dict())

print("\nMemulai Training Sub-Model Organik (MobileNetV3)")
for epoch in range(EPOCHS_ORG):
    print(f'Epoch {epoch+1}/{EPOCHS_ORG}')
    
    # Fase Training
    model_organic.train()
    running_loss, running_corrects = 0.0, 0
    tqdm_train = tqdm(train_loader_org, desc="Org Train", leave=False)
    for inputs, labels in tqdm_train:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_org.zero_grad()
        outputs = model_organic(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion_org(outputs, labels)
        loss.backward()
        optimizer_org.step()
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
    epoch_loss_train = running_loss / len(train_dataset_org)
    epoch_acc_train = running_corrects.double() / len(train_dataset_org)
    
    # Fase Validasi
    model_organic.eval()
    running_loss_val, running_corrects_val = 0.0, 0
    tqdm_val = tqdm(val_loader_org, desc="Org Val", leave=False)
    with torch.no_grad():
        for inputs, labels in tqdm_val:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_organic(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion_org(outputs, labels)
            running_loss_val += loss.item() * inputs.size(0)
            running_corrects_val += torch.sum(preds == labels.data)
    epoch_loss_val = running_loss_val / len(val_dataset_org)
    epoch_acc_val = running_corrects_val.double() / len(val_dataset_org)
    
    print(f"[TRAIN] Loss: {epoch_loss_train:.4f} Acc: {epoch_acc_train:.4f} | [VAL] Loss: {epoch_loss_val:.4f} Acc: {epoch_acc_val:.4f}")
    
    if epoch_acc_val > best_acc_org:
        best_acc_org = epoch_acc_val
        best_wts_org = copy.deepcopy(model_organic.state_dict())
        print(f"Akurasi naik ke {best_acc_org:.4f}. hasil diupdate")


Memulai Training Sub-Model Organik (MobileNetV3)
Epoch 1/10


Org Train:   0%|          | 0/26 [00:00<?, ?it/s]

Org Val:   0%|          | 0/6 [00:00<?, ?it/s]

[TRAIN] Loss: 0.3859 Acc: 0.8873 | [VAL] Loss: 0.1780 Acc: 0.9325
Akurasi naik ke 0.9325. hasil diupdate
Epoch 2/10


Org Train:   0%|          | 0/26 [00:00<?, ?it/s]

...

Org Val:   0%|          | 0/6 [00:00<?, ?it/s]

[TRAIN] Loss: 0.0042 Acc: 0.9988 | [VAL] Loss: 0.0070 Acc: 0.9939
Epoch 10/10


Org Train:   0%|          | 0/26 [00:00<?, ?it/s]

Org Val:   0%|          | 0/6 [00:00<?, ?it/s]

[TRAIN] Loss: 0.0029 Acc: 1.0000 | [VAL] Loss: 0.0138 Acc: 0.9939


### **Save Model Organik**

In [13]:
model_organic.load_state_dict(best_wts_org)
os.makedirs("../saved_models", exist_ok=True)
torch.save(model_organic.state_dict(), "../saved_models/sub_organic_best.pth")
print("Sub-model Organik berhasil disimpan!")

Sub-model Organik berhasil disimpan!


# **Sub Model Anorganik ((ConvNeXt-Tiny)**)

### **Load Data Anorganik**

In [10]:
inorg_dir = os.path.join(BASE_SUB_DIR, "inorganic_branch")
train_dataset_inorg = datasets.ImageFolder(os.path.join(inorg_dir, "train"), data_transforms['train'])
val_dataset_inorg = datasets.ImageFolder(os.path.join(inorg_dir, "val"), data_transforms['val_test'])

train_loader_inorg = DataLoader(train_dataset_inorg, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader_inorg = DataLoader(val_dataset_inorg, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

inorg_classes = train_dataset_inorg.classes
print(f"Kelas Cabang Anorganik ({len(inorg_classes)} kelas): {inorg_classes}")

Kelas Cabang Anorganik (7 kelas): ['Cardboard', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash']


### **Build Model**

In [11]:
print("Memuat pre-trained ConvNeXt-Tiny...")
model_inorganic = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
num_features_inorg = model_inorganic.classifier[2].in_features
model_inorganic.classifier[2] = nn.Linear(num_features_inorg, len(inorg_classes))
model_inorganic = model_inorganic.to(device)

Memuat pre-trained ConvNeXt-Tiny...


### **Loss dan Optimizer**

In [17]:
criterion_inorg = nn.CrossEntropyLoss()
optimizer_inorg = optim.Adam(model_inorganic.parameters(), lr=5e-5)

### **Train Function**

In [18]:
EPOCHS_INORG = 10 
best_acc_inorg = 0.0
best_wts_inorg = copy.deepcopy(model_inorganic.state_dict())

print("\nMemulai Training Sub-Model Anorganik (ConvNeXt-Tiny)")
for epoch in range(EPOCHS_INORG):
    print(f'Epoch {epoch+1}/{EPOCHS_INORG}')
    
    # Fase Training
    model_inorganic.train()
    running_loss, running_corrects = 0.0, 0
    tqdm_train = tqdm(train_loader_inorg, desc="Inorg Train", leave=False)
    for inputs, labels in tqdm_train:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_inorg.zero_grad()
        outputs = model_inorganic(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion_inorg(outputs, labels)
        loss.backward()
        optimizer_inorg.step()
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
    epoch_loss_train = running_loss / len(train_dataset_inorg)
    epoch_acc_train = running_corrects.double() / len(train_dataset_inorg)
    
    # Fase Validasi
    model_inorganic.eval()
    running_loss_val, running_corrects_val = 0.0, 0
    tqdm_val = tqdm(val_loader_inorg, desc="Inorg Val", leave=False)
    with torch.no_grad():
        for inputs, labels in tqdm_val:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_inorganic(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion_inorg(outputs, labels)
            running_loss_val += loss.item() * inputs.size(0)
            running_corrects_val += torch.sum(preds == labels.data)
    epoch_loss_val = running_loss_val / len(val_dataset_inorg)
    epoch_acc_val = running_corrects_val.double() / len(val_dataset_inorg)
    
    print(f"[TRAIN] Loss: {epoch_loss_train:.4f} Acc: {epoch_acc_train:.4f} | [VAL] Loss: {epoch_loss_val:.4f} Acc: {epoch_acc_val:.4f}")
    
    if epoch_acc_val > best_acc_inorg:
        best_acc_inorg = epoch_acc_val
        best_wts_inorg = copy.deepcopy(model_inorganic.state_dict())
        print(f"Akurasi naik ke {best_acc_inorg:.4f}. hasil diupdate")


Memulai Training Sub-Model Anorganik (ConvNeXt-Tiny)
Epoch 1/10


Inorg Train:   0%|          | 0/117 [00:00<?, ?it/s]

Inorg Val:   0%|          | 0/23 [00:00<?, ?it/s]

[TRAIN] Loss: 0.9688 Acc: 0.6933 | [VAL] Loss: 0.4445 Acc: 0.8677
Akurasi naik ke 0.8677. hasil diupdate
Epoch 2/10


Inorg Train:   0%|          | 0/117 [00:00<?, ?it/s]

...

Inorg Val:   0%|          | 0/23 [00:00<?, ?it/s]

[TRAIN] Loss: 0.0211 Acc: 0.9957 | [VAL] Loss: 0.0322 Acc: 0.9877
Epoch 10/10


Inorg Train:   0%|          | 0/117 [00:00<?, ?it/s]

Inorg Val:   0%|          | 0/23 [00:00<?, ?it/s]

[TRAIN] Loss: 0.0109 Acc: 0.9984 | [VAL] Loss: 0.0381 Acc: 0.9850


### **Save Model Anorganik**

In [19]:
model_inorganic.load_state_dict(best_wts_inorg)
torch.save(model_inorganic.state_dict(), "../saved_models/sub_inorganic_best.pth")
print("Sub-model Anorganik berhasil disimpan!")

Sub-model Anorganik berhasil disimpan!
